# 🧪 Thực Nghiệm Toàn Diện: Bộ 5 Bài Test Khoa Học Chứng Minh Điểm Yếu Của LiDAR Trên Kaggle (2x GPU Tesla T4 16GB)
### **Khung Lý Thuyết**: Dimension-Free Lipschitz Bound, Particle Filtering (SMC) & Randomized Smoothing (RS-LiDAR)

Notebook này thiết lập môi trường thực nghiệm hoàn chỉnh để kiểm chứng toàn bộ **Bộ 5 Bài Test Khoa Học Độc Lập** so sánh giữa **LiDAR gốc (ICML 2026)** và **RS-LiDAR (Smoothed Surrogate)**:

---

### 📌 Hệ Thống 5 Bài Test Khoa Học:
1. **🧪 TEST 1: Kháng Sai số Bộ giải (Solver Error Robustness & Theorem 1)**
   * *Lý thuyết*: Khi dùng DPM-Solver 5 bước, sai số hình học $\mathbf{e}_i = \hat{\mathbf{x}}_0^i - \mathbf{x}_0^i$ khiến hàm thưởng thô $r(\hat{\mathbf{x}}_0)$ bùng nổ do $L_0 \to \infty$. RS-LiDAR chặn trên sai số bằng hằng số Lipschitz hữu hạn: $|r_\sigma(\hat{\mathbf{x}}_0) - r_\sigma(\mathbf{x}_0)| \le L_\sigma \|\mathbf{e}_i\|_2$.
   * *Chỉ số đo*: Sai số điểm thưởng $|\Delta r|$ và hệ số tương quan thứ bậc Kendall $\tau$ giữa 5 bước DPM và 50 bước DDIM chuẩn trên đa mô hình reward (ImageReward, CLIP-Score, HPS v2.1, Aesthetic Score, PickScore).

2. **🧪 TEST 2: Kháng Sụp đổ Trọng số Softmax (Softmax Mode Collapse Prevention)**
   * *Lý thuyết*: Hàm $\exp(\lambda r)$ của LiDAR với $\lambda=5000$ bị bão hòa One-Hot tại các đỉnh gai nhọn cục bộ, dồn $99\%$ trọng số vào đúng 1 hạt (Best-of-1 Trap). RS-LiDAR làm phẳng các đỉnh nhọn, duy trì phân bổ trọng số đa hạt mượt mà.
   * *Chỉ số đo*: Entropy Shannon $H(w^r) = -\sum w_i^r \log_2 w_i^r$ xuyên suốt 50 bước khử nhiễu $t \in [1000, 0]$.

3. **🧪 TEST 3: Kháng Rung lắc Vector Dẫn đường (Guidance Field Lipschitz Stability)**
   * *Lý thuyết*: Vector dẫn đường $\mathbf{g}_t(\mathbf{x}_t)$ của LiDAR bị bẻ ngoặt hỗn loạn khi trạng thái $\mathbf{x}_t$ dao động nhỏ $\delta$. RS-LiDAR đảm bảo độ nhạy ma trận Jacobi $\|\frac{\partial \mathbf{g}_t}{\partial \mathbf{x}_t}\| \le C \cdot L_\sigma < \infty$.
   * *Chỉ số đo*: Độ ổn định góc quay $\text{CosSim}(\mathbf{g}_t, \mathbf{g}_{t+\delta})$ tại nhiễu vi mô $\|\delta\|_2 = 10^{-3}$.

4. **🧪 TEST 4: Đo Mức Độ Suy Thoái Hạt Hữu Hiệu (Effective Sample Size - ESS & Particle Starvation)**
   * *Lý thuyết (Sequential Monte Carlo / Particle Filter)*: Đo số hạt hữu hiệu $\text{ESS}_t = \frac{1}{\sum (w_i^r)^2}$. LiDAR bị sụp đổ $\text{ESS} \approx 1.05 - 1.20$, bóc trần sự thật rằng 98% chi phí tính toán của 50 hạt bị lãng phí. RS-LiDAR duy trì $\text{ESS} \ge 15 - 30$, kích hoạt sức mạnh đa hạt thực thụ.
   * *Chỉ số đo*: $\text{ESS}_t$, Normalized ESS ($\text{NESS}$), trọng số lớn nhất $w_{\max}$ và số hạt tích cực $N_{\text{active}}$.

5. **🧪 TEST 5: Khảo Sát Giới Hạn Bước Bộ Giải Nhanh (Step-Budget Solver Scaling: $S \in \{2, 3, 5, 8, 15\}$)**
   * *Lý thuyết*: LiDAR chọn $S=5$ theo cảm tính. Khi giảm số bước $S < 5$ để tăng tốc suy luận, sai số bộ giải $\|\mathbf{e}_S\|_2$ tăng vọt khiến LiDAR sụp đổ thứ bậc $\tau$ thẳng đứng. Nhờ chặn Lipschitz Định lý 1, RS-LiDAR tại **$S=3$ bước** vẫn đạt độ chính xác tương đương hoặc vượt trội LiDAR gốc ở $S=5$ bước $\implies$ Giúp tăng tốc độ sinh ảnh gấp gần 2 lần!
   * *Chỉ số đo*: Đường cong thoái hóa bước $\tau(S)$ và sai số $|\Delta r(S)|$ so với chuẩn 50 bước DDIM.

## 1. Kiểm Tra Phần Cứng 2x GPU Tesla T4 (16GB VRAM) & Cấu Hình Bộ Nhớ
Bật chế độ `expandable_segments:True` để chống phân mảnh bộ nhớ VRAM khi chạy đồng thời nhiều reward model.

In [ ]:
import os, sys, torch

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA khả dụng: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f"🎮 Số lượng GPU phát hiện: {n_gpus}")
    for i in range(n_gpus):
        vram = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f"  • GPU {i}: {torch.cuda.get_device_name(i)} ({vram:.2f} GB VRAM)")
else:
    raise RuntimeError("❌ Không phát hiện GPU CUDA! Vui lòng bật GPU trong Settings -> Accelerator -> GPU T4 x2.")

# Chống phân mảnh bộ nhớ CUDA trên GPU T4 16GB (Issue 19)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
!nvidia-smi

## 2. Thiết Lập Mã Nguồn Repo & Cài Đặt Bộ Thư Viện Đa Hàm Thưởng
Cài đặt toàn bộ 5 mô hình đánh giá: **ImageReward, CLIP-Score, HPS v2.1, Aesthetic Score, PickScore** và tải trước trọng số chống lỗi đa tiến trình.

In [ ]:
import os, shutil, glob, subprocess, threading, urllib.request

# 1. Đồng bộ repo
REPO_DIR = "/kaggle/working/RS-LiDAR"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/leekwanreal/RS-LiDAR.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull origin main

WORKDIR = f"{REPO_DIR}/Diffusion-LiDAR-Sampling" if os.path.exists(f"{REPO_DIR}/Diffusion-LiDAR-Sampling") else REPO_DIR
os.chdir(WORKDIR)
%cd {WORKDIR}
print("📂 Thư mục làm việc:", os.getcwd())

OUTPUT_DIR = "/kaggle/working/test_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("📁 Thư mục lưu kết quả thí nghiệm:", OUTPUT_DIR)

# 2. Hàm tiện ích chạy song song 2 GPU hiển thị log trực tiếp
def run_commands_parallel(cmd0, cmd1):
    p0 = subprocess.Popen(cmd0, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    p1 = subprocess.Popen(cmd1, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    def stream_logs(proc, prefix):
        for line in iter(proc.stdout.readline, ''):
            if line.strip():
                print(f"{prefix} {line.strip()}")
        proc.stdout.close()
    t0 = threading.Thread(target=stream_logs, args=(p0, "[GPU 0]"))
    t1 = threading.Thread(target=stream_logs, args=(p1, "[GPU 1]"))
    t0.start(); t1.start()
    t0.join(); t1.join()
    p0.wait(); p1.wait()

# 3. Cấu hình biến môi trường
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

# 4. Cài đặt các thư viện
!pip install -q --upgrade protobuf
!pip install -q transformers==4.38.2 diffusers==0.31.0 accelerate==1.2.1 safetensors huggingface-hub einops ftfy timm peft
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q git+https://github.com/THUDM/ImageReward.git
!pip install -q hpsv2 matplotlib tqdm scipy seaborn pandas tabulate

# 5. Tải trước vocab và checkpoint HPSv2.1 để chống lỗi tải đồng thời trên 2 GPU
try:
    import hpsv2
    hpsv2_vocab = os.path.join(os.path.dirname(hpsv2.__file__), "src", "open_clip", "bpe_simple_vocab_16e6.txt.gz")
    os.makedirs(os.path.dirname(hpsv2_vocab), exist_ok=True)
    if not os.path.exists(hpsv2_vocab):
        urllib.request.urlretrieve("https://github.com/openai/CLIP/raw/main/clip/bpe_simple_vocab_16e6.txt.gz", hpsv2_vocab)
    
    from huggingface_hub import hf_hub_download
    hps_cache = os.path.expanduser("~/.cache/hpsv2")
    os.makedirs(hps_cache, exist_ok=True)
    hps_ckpt = os.path.join(hps_cache, "HPS_v2.1_compressed.pt")
    if not os.path.exists(hps_ckpt) or os.path.getsize(hps_ckpt) < 1000000:
        print("⏳ Đang tải trước trọng số HPSv2.1...")
        try:
            hf_hub_download(repo_id="xswu/HPSv2", filename="HPS_v2.1_compressed.pt", local_dir=hps_cache)
            print("✅ Đã tải xong HPSv2.1!")
        except Exception:
            urllib.request.urlretrieve("https://huggingface.co/xswu/HPSv2/resolve/main/HPS_v2.1_compressed.pt", hps_ckpt)
            print("✅ Đã tải xong HPSv2.1!")
except Exception as e:
    print(f"Lưu ý khởi tạo HPS: {e}")

print("\n✅ Môi trường cho Bộ 5 Bài Test đã sẵn sàng 100%!")

## 3. [SIÊU TỐC: ~15 GIÂY] Chạy Riêng Bài Test 4 (Đo ESS & Sụp Đổ Hạt)
Đo **Effective Sample Size (ESS)**, Normalized ESS, và trọng số cực đại $w_{\max}$ theo chuẩn Sequential Monte Carlo để chứng minh hiện tượng Best-of-1 Trap của LiDAR.

In [ ]:
import os, glob, pandas as pd
from IPython.display import display, Markdown

!python test_lidar_weaknesses.py \
    --test 4 \
    --num_particles 50 \
    --sigma 0.25 \
    --tune_sigma \
    --sigmas "0.10,0.25,0.50,1.00" \
    --output_dir "{OUTPUT_DIR}"

# Hiển thị kết quả Test 4 nếu có
t4_files = glob.glob(f"{OUTPUT_DIR}/test_4_*.csv")
if t4_files:
    df_t4 = pd.read_csv(t4_files[0])
    print("\n📊 KẾT QUẢ BÀI TEST 4 (ESS & PARTICLE STARVATION):")
    display(df_t4)

## 4. [SIÊU TỐC: ~10 GIÂY] Chạy Riêng Bài Test 2 & Test 3 (Entropy & Cosine Stability)
Cả 2 bài test này hoàn toàn là các phép toán ma trận tensor PyTorch trên latent (không chạy diffusion UNet, không decode VAE).

In [ ]:
print("🔬 Đang chạy Test 2 (Entropy Softmax)...")
!python test_lidar_weaknesses.py --test 2 --output_dir "{OUTPUT_DIR}"

print("\n🔬 Đang chạy Test 3 (Độ ổn định Lipschitz của vector dẫn đường)...")
!python test_lidar_weaknesses.py --test 3 --output_dir "{OUTPUT_DIR}"

## 5. [NHANH: ~5 PHÚT] Chạy Riêng Bài Test 5 (Khảo Sát Bước Bộ Giải $S \in \{2, 3, 5, 8, 15\}$)
Kiểm chứng Định lý 1 Dimension-Free Lipschitz Bound, chứng minh RS-LiDAR tại $S=3$ đạt độ chính xác ngang ngửa hoặc vượt trội LiDAR tại $S=5$.

In [ ]:
n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1

if n_gpus >= 2:
    print("🚀 [2 GPU] Đang chạy Test 5 song song trên 2 GPU T4...")
    cmd0 = f"python test_lidar_weaknesses.py --test 5 --num_prompts 10 --num_particles 10 --sigma 0.05 --num_shards 2 --shard_id 0 --output_dir '{OUTPUT_DIR}'"
    cmd1 = f"python test_lidar_weaknesses.py --test 5 --num_prompts 10 --num_particles 10 --sigma 0.05 --num_shards 2 --shard_id 1 --output_dir '{OUTPUT_DIR}'"
    run_commands_parallel(cmd0, cmd1)
else:
    print("🚀 [1 GPU] Đang chạy Test 5 trên 1 GPU...")
    !python test_lidar_weaknesses.py --test 5 --num_prompts 10 --num_particles 10 --sigma 0.05 --output_dir "{OUTPUT_DIR}"

## 6. [TOÀN DIỆN] Chạy Toàn Bộ 5 Bài Test Khoa Học Song Song Trên 2 GPU (`--test all`)
Chạy trọn gói từ Test 1 đến Test 5 trên cả 5 hàm thưởng (ImageReward, CLIP-Score, HPS v2.1, Aesthetic Score, PickScore) và 6 mốc $\sigma \in [0.05, 1.00]$, tự động gộp checkpoint thành bảng so sánh hoàn chỉnh.

In [ ]:
n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1

if n_gpus >= 2:
    print("🚀 [2 GPU] Đang chạy trọn bộ 5 bài test song song trên GPU 0 và GPU 1...")
    cmd0 = f"""python test_lidar_weaknesses.py \
        --test all --num_prompts 20 --num_particles 20 \
        --sigma 0.05 --tune_sigma --sigmas "0.05,0.10,0.15,0.25,0.50,1.00" \
        --all_rewards --num_shards 2 --shard_id 0 --output_dir "{OUTPUT_DIR}""""
        
    cmd1 = f"""python test_lidar_weaknesses.py \
        --test all --num_prompts 20 --num_particles 20 \
        --sigma 0.05 --tune_sigma --sigmas "0.05,0.10,0.15,0.25,0.50,1.00" \
        --all_rewards --num_shards 2 --shard_id 1 --output_dir "{OUTPUT_DIR}""""
        
    run_commands_parallel(cmd0, cmd1)
    print("✅ Cả 2 GPU đã hoàn thành toàn bộ 5 bài test!")
else:
    print("🚀 [1 GPU] Đang chạy toàn bộ 5 bài test trên 1 GPU...")
    !python test_lidar_weaknesses.py \
        --test all --num_prompts 20 --num_particles 20 \
        --sigma 0.05 --tune_sigma --sigmas "0.05,0.10,0.15,0.25,0.50,1.00" \
        --all_rewards --output_dir "{OUTPUT_DIR}"

## 7. Hiển Thị Bảng Kết Quả Khoa Học & Toàn Bộ Đồ Thị Trực Quan
Render trực quan bảng so sánh 5 bài test, bảng khảo sát $\sigma$ ablation, và các biểu đồ xuất bản chất lượng cao.

In [ ]:
import pandas as pd, glob, os
from IPython.display import display, Image, Markdown

# 1. Hiển thị Bảng so sánh khoa học các bài test
table_files = glob.glob(f"{OUTPUT_DIR}/*comparison*.csv")
if table_files:
    df = pd.read_csv(table_files[0])
    print("📊 BẢNG TỔNG HỢP KẾT QUẢ CÁC BÀI TEST KHOA HỌC:")
    display(df)
    
    md_files = glob.glob(f"{OUTPUT_DIR}/*comparison*.md")
    if md_files:
        with open(md_files[0], 'r', encoding='utf-8') as f:
            display(Markdown(f.read()))

# 2. Hiển thị Bảng khảo sát tham số Sigma Ablation
sigma_files = glob.glob(f"{OUTPUT_DIR}/*sigma*.csv")
if sigma_files:
    df_sigma = pd.read_csv(sigma_files[0])
    print("\n📈 BẢNG KHẢO SÁT THAM SỐ BÁN KÍNH LÀM MỊN SIGMA:")
    display(df_sigma)

# 3. Hiển thị tất cả các biểu đồ khoa học đã xuất
png_files = sorted(glob.glob(f"{OUTPUT_DIR}/*.png"))
for p in png_files:
    print(f"\n🖼️ Đồ thị: {os.path.basename(p)}")
    display(Image(filename=p))

## 8. Đóng Gói Toàn Bộ Báo Cáo & Đồ Thị Kết Quả (1-Click Download)
Nén toàn bộ bảng biểu CSV, Markdown và ảnh biểu đồ thành `weakness_proof_results.zip` tại `/kaggle/working/` để tải về dễ dàng từ tab **Output** của Kaggle.

In [ ]:
import os

zip_name = "/kaggle/working/weakness_proof_results.zip"
print(f"📦 Đang nén dữ liệu kết quả vào {zip_name}...")

!zip -r -q {zip_name} {OUTPUT_DIR} \
    /kaggle/working/*.csv \
    /kaggle/working/*.md \
    /kaggle/working/*.png 2>/dev/null || true

if os.path.exists(zip_name):
    size_mb = os.path.getsize(zip_name) / (1024 * 1024)
    print(f"🎉 ĐÓNG GÓI THÀNH CÔNG! Dung lượng file: {size_mb:.2f} MB")
    print(f"📁 Đường dẫn file zip: {zip_name}")
    print("👉 Bạn có thể tải file này từ tab 'Output' của giao diện Kaggle!")
else:
    print("⚠️ Không tìm thấy file zip được tạo.")